# 15 分钟入门 Julia 与自动微分

很大程度上参考了 [用 Julia 在 10 分钟内理解自动微分](https://youtu.be/vAp6nUMrKYg)

## 1- 对偶数

[来源](https://en.wikipedia.org/wiki/Automatic_differentiation#Automatic_differentiation_using_dual_numbers)

在线性代数中，对偶数通过添加一个新元素 $\varepsilon$（epsilon）来扩展实数，这个元素满足 $\varepsilon^2 = 0$。因此对偶数的乘法定义为：
$$(a+b\varepsilon )(c+d\varepsilon )=ac+(ad+bc)\varepsilon,$$
加法按分量进行：
$$(a+b\varepsilon )+(c+d\varepsilon )=(a+c)+(b+d)\varepsilon.$$

使用 [Julia](https://docs.julialang.org/en/v1/)，我们可以像下面这样为对偶数创建一种数据类型：


In [ ]:
struct Dual <: Number
    value::Float64
    epsilon::Float64
end

现在为对偶数定义加法、减法和乘法。为此，我们给 [Julia Base](https://docs.julialang.org/en/v1/base/base/) 的运算符 `+,-,*` 定义新的[方法](https://docs.julialang.org/en/v1/manual/methods/)。


In [ ]:
import Base: +, -, *
+(z::Dual, w::Dual) = Dual(z.value+w.value, z.epsilon+w.epsilon)
-(z::Dual, w::Dual) = Dual(z.value-w.value, z.epsilon-w.epsilon)
*(z::Dual, w::Dual) = Dual(z.value*w.value, z.value*w.epsilon+z.epsilon*w.value)

为了方便，我们再为对偶数定义一个新的显示函数。


In [ ]:
Base.show(io::IO,x::Dual) = print(io,x.value," + ",x.epsilon," ε")

In [ ]:
a=Dual(2,1)

现在我们准备好开始玩对偶数了。


In [ ]:
a*a

In [ ]:
a^2

In [ ]:
a^7

最后这两条命令相当出人意料，因为我们从没定义过对偶数的幂运算！但事实证明，在 Julia 里，任何支持 `*` 的 `x` 都定义了幂运算，这可以从[这里](https://github.com/JuliaLang/julia/blob/44fa15b1502a45eac76c9017af94332d4557b251/base/intfuncs.jl#L188)看到，或者跟随下面这条命令给出的链接：


In [ ]:
@which a^7

In [ ]:
2*a

要修正这个问题，我们需要：
- 把实数 `2` 转换成 `Dual(2,0)`
- 告诉 Julia 在每次出现对偶数与实数混合的表达式时自动做这种转换

这就是所谓的[转换与提升（conversion and promotion）](https://docs.julialang.org/en/v1/manual/conversion-and-promotion/#conversion-and-promotion)，可以像下面这样做：


In [ ]:
import Base: convert, promote_rule
convert(::Type{Dual}, x::Real) = Dual(x,zero(x))
promote_rule(::Type{Dual}, ::Type{<:Number}) = Dual

In [ ]:
2*a

In [ ]:
a-1

In [ ]:
b=Dual(1,1)
3*(a+b)^2

## 2- 多项式的自动微分

现在我们能对多项式求导了。考虑 $P(x) = p_0+p_1x+p_2x^2+\dots +p_n x^n$，注意由于 $\varepsilon^2 =0$（$\varepsilon$ 是幂零元），我们有 $(x+\varepsilon y)^k = x^k + kx^{k-1}\varepsilon y$。因此
$$
P(x+\varepsilon y) = P(x) +  \left(p_1 + 2p_2 x+ \dots np_n x^{n-1}\right)y\varepsilon = P(x) + P'(x)y\varepsilon 
$$


In [ ]:
a

In [ ]:
3*(1+a)^2

In [ ]:
value(z::Dual) = z.value
epsilon(z::Dual) = z.epsilon

In [ ]:
epsilon(3*(1+a)^2)

现在我们可以定义一个简单的[函数](https://docs.julialang.org/en/v1/manual/functions/)，计算多项式在给定点的导数，如下：


In [ ]:
function derivative(f,x::Real)
    epsilon(f(Dual(x,1)))
end

In [ ]:
derivative(x->1+x+3x^2,1)

## 3- 更进一步：处理 $\sqrt{}$


In [ ]:
a^(1/2)

要处理这个问题，我们按照这个不错的[教程](https://github.com/JuliaAcademy/JuliaTutorials/blob/master/introductory-tutorials/intro-to-julia/AutoDiff.ipynb)里描述的巴比伦算法来做：
> 重复 $ t \leftarrow  \frac{1}{2}\left(t+\frac{x}{t}\right)$ 直到 $t$ 收敛到 $\sqrt{x}$。


In [ ]:
function Babylonian(x; N = 10) 
    t = (1+x)/2
    for i = 2:N; t=(t+x/t)/2  end    
    t
end

In [ ]:
Babylonian(2), √2  # 输入 \sqrt+<tab> 得到这个符号

巴比伦算法只用到加法和除法，所以我们需要为对偶数定义除法：
$$\frac{a+b\varepsilon}{c+d\varepsilon}= \frac{a}{c}\left(1+\frac{b}{a}\varepsilon\right)\left(1-\frac{d}{c}\varepsilon\right)= \frac{a}{c} +\frac{bc-ad}{c^2}\varepsilon.$$


In [ ]:
import Base:/
/(x::Dual, y::Dual) = Dual(x.value/y.value, (y.value*x.epsilon - x.value*y.epsilon)/y.value^2)

In [ ]:
a

In [ ]:
1/(1+a)

In [ ]:
Babylonian(a), √2, 0.5/√2

In [ ]:
derivative(x->Babylonian(x),2)

In [ ]:
function dBabylonian(x; N = 10) 
    t = (1+x)/2
    dt = 1/2
    for i = 1:N;  
        t = (t+x/t)/2; 
        dt = (dt+(t-x*dt)/t^2)/2; 
    end    
    dt
end  

In [ ]:
x = 2; dBabylonian(x), .5/√x

In [ ]:
derivative(x->Babylonian(x)^4,4)

In [ ]:
derivative(x->(1+3*Babylonian(x))^3/Babylonian(x),2)

In [ ]:
# using Pkg; Pkg.add("ForwardDiff")
using ForwardDiff

In [ ]:
ForwardDiff.derivative(sqrt, 2)

In [ ]:
ForwardDiff.derivative(Babylonian, 2)

In [ ]:
ForwardDiff.derivative(x->(1+3*sqrt(x))^3/sqrt(x),2)

想了解更多对偶数，可以看看这个（已不再维护的）Julia 包 [DualNumbers.jl](https://github.com/JuliaDiff/DualNumbers.jl)
